# analyze_relation

지정한 관계 `a`를 가지는 노드 타입 `b`(source) → `c`(target) 페어를 모두 출력합니다.

**노드 타입**: `c`, `s`, `m`, `t`

**관계 (relation)**:
- `SOURCE` — `src_out` 에지 (라벨 없음)
- `SUPPORT`, `CONTRADICT`, `SHIFT_TO`, `IRRELEVANT` — `evid_out` 에지 라벨

In [ ]:
import json
from collections import Counter
from pathlib import Path

# Relative to gmem6/ (where this notebook lives)
GRAPH_PATH = Path(
    'config_0_outputs_Qwen3-1.7B_opposed/session_0_49/'
    'memory_snapshots/session_0/graph/graph.json'
)

with open(GRAPH_PATH) as f:
    G = json.load(f)

NODES = G['nodes']
SRC_OUT = G['src_out']
EVID_OUT = G['evid_out']

print(f'nodes: {len(NODES)}')
print('node_type 분포:', Counter(n['node_type'] for n in NODES.values()))
print('src_out edges:', sum(len(v) for v in SRC_OUT.values()))
evid_labels = Counter()
for tgts in EVID_OUT.values():
    for lbl in tgts.values():
        evid_labels[lbl] += 1
print('evid_out 라벨별 edges:', dict(evid_labels))

nodes: 1410
node_type 분포: Counter({'c': 656, 's': 589, 'm': 110, 't': 55})
src_out edges: 2600
evid_out 라벨별 edges: {'IRRELEVANT': 229, 'SUPPORT': 3442, 'SHIFT_TO': 26, 'CONTRADICT': 48}


In [ ]:
VALID_NODE_TYPES = {'c', 's', 'e', 't'}
EVID_LABELS = {'SUPPORT', 'CONTRADICT', 'SHIFT_TO', 'IRRELEVANT'}


def find_pairs(relation: str, src_type: str, tgt_type: str):
    rel = relation.upper()
    if src_type not in VALID_NODE_TYPES or tgt_type not in VALID_NODE_TYPES:
        raise ValueError(f'node type은 {VALID_NODE_TYPES} 중 하나여야 합니다.')

    pairs = []
    if rel == 'SOURCE':
        for src_id, tgt_ids in SRC_OUT.items():
            if NODES[src_id]['node_type'] != src_type:
                continue
            for tgt_id in tgt_ids:
                if NODES[tgt_id]['node_type'] == tgt_type:
                    pairs.append((src_id, tgt_id))
    elif rel in EVID_LABELS:
        for src_id, tgt_map in EVID_OUT.items():
            if NODES[src_id]['node_type'] != src_type:
                continue
            for tgt_id, lbl in tgt_map.items():
                if lbl == rel and NODES[tgt_id]['node_type'] == tgt_type:
                    pairs.append((src_id, tgt_id))
    else:
        raise ValueError(
            f'relation은 SOURCE | {" | ".join(sorted(EVID_LABELS))} 중 하나여야 합니다.'
        )
    return pairs


def short(text: str, n: int | None = 80) -> str:
    text = text.replace('\n', ' ')
    if n is None or len(text) <= n:
        return text
    return text[:n] + '...'


def print_pairs(relation: str, src_type: str, tgt_type: str, show_content: bool = True,
                limit: int | None = None, content_len: int | None = 80):
    pairs = find_pairs(relation, src_type, tgt_type)
    print(f'[{relation}] ({src_type} → {tgt_type}): {len(pairs)} pairs')
    print('=' * 80)
    for i, (s, t) in enumerate(pairs):
        if limit is not None and i >= limit:
            print(f'... ({len(pairs) - limit} more)')
            break
        if show_content:
            print(f'[{i+1}]')
            print(f'    src({src_type}): {short(NODES[s]["content"], content_len)}')
            print(f'    tgt({tgt_type}): {short(NODES[t]["content"], content_len)}')
        else:
            print(f'[{i+1}]')
    return pairs

## 사용 예시

여기서 `RELATION`, `SRC_TYPE`, `TGT_TYPE` 만 바꿔서 실행하세요.

In [ ]:
RELATION = 'SUPPORT'   # SOURCE | SUPPORT | CONTRADICT | SHIFT_TO | IRRELEVANT
SRC_TYPE = 's'         # c | s | m | t
TGT_TYPE = 'c'         # c | s | m | t

pairs = print_pairs(RELATION, SRC_TYPE, TGT_TYPE, show_content=True, limit=20, content_len=None)

[SUPPORT] (s → c): 0 pairs


## 전체 조합 요약 (선택)

각 relation × (src_type → tgt_type) 조합별 페어 개수를 한 번에 확인.

In [ ]:
RELATIONS = ['SOURCE', 'SUPPORT', 'CONTRADICT', 'SHIFT_TO', 'IRRELEVANT']
TYPES = ['c', 's', 'e', 't']

print(f'{"relation":<11} {"src→tgt":<8} {"count":>6}')
print('-' * 30)
for rel in RELATIONS:
    for b in TYPES:
        for c in TYPES:
            n = len(find_pairs(rel, b, c))
            if n:
                print(f'{rel:<11} {b}→{c:<6} {n:>6}')

relation    src→tgt   count
------------------------------
SOURCE      c→s         589
SOURCE      c→m         656
SOURCE      c→t         656
SOURCE      s→t         589
SOURCE      m→t         110
SUPPORT     s→s        1750
SUPPORT     s→t         833
SUPPORT     m→s         597
SUPPORT     m→m          82
SUPPORT     m→t         165
SUPPORT     t→t          15
CONTRADICT  s→s          36
CONTRADICT  s→t          11
CONTRADICT  m→s           1
SHIFT_TO    s→s          13
SHIFT_TO    t→t          13
IRRELEVANT  s→s          61
IRRELEVANT  s→t         119
IRRELEVANT  m→s          11
IRRELEVANT  m→m          27
IRRELEVANT  m→t          11


In [ ]:
SESSION_NUM = 0    # e.g. 0, 5
CALL_KEY    = '6' # e.g. '2', '2b', '5a'  → matches folder call_{CALL_KEY}_*

# ── resolve path ─────────────────────────────────────────────────────────────
session_dir = GRAPH_PATH.parents[3]          # config_N_.../session_X_Y/
prompt_log  = session_dir / 'prompt_log' / f'session_{SESSION_NUM}'
matches     = [d for d in prompt_log.iterdir() if d.name.startswith(f'call_{CALL_KEY}_')]

if not matches:
    raise FileNotFoundError(f'call_{CALL_KEY}_* 폴더를 찾을 수 없습니다: {prompt_log}')
call_dir = sorted(matches)[0]
calls_path = call_dir / 'calls.jsonl'

# ── load & print ──────────────────────────────────────────────────────────────
with open(calls_path) as f:
    records = [json.loads(line) for line in f if line.strip()]

print(f'파일: {calls_path}')
print(f'records: {len(records)}\n')
for i, rec in enumerate(records):
    print(f'{"─"*80}')
    print(f'[{i}] call_type: {rec.get("call_type", "?")}')
    #print(f'\n■ system_prompt\n{rec["system_prompt"]}')
    print(f'\n■ user_prompt\n{rec["user_prompt"]}')

파일: config_0_outputs_Qwen3-1.7B_opposed/session_0_49/prompt_log/session_0/call_6_qa/calls.jsonl
records: 5

────────────────────────────────────────────────────────────────────────────────
[0] call_type: call_6_qa

■ user_prompt
[Question]
How can I find reliable tech bloggers to recommend educational apps for my children?

[Retrieved Memory]
[Current Constraints]
(These are HIGH recall-priority user states the assistant should honor in the response, unless the user explicitly overrides them in the current message.)
o ago] The user is looking for high-quality educational technology tools for their children and is concerned about trustworthiness of recommendations.
o ago] The user is interested in using educational apps in their class.
o ago] The user is considering starting their own channel and sharing their thoughts and insights with others.
[21d ago] The user prefers outdoor activities and is looking for a competitive hobby.
[6d ago] The user is passionate about swimming and has bee

## QA prompt 출력 셀

In [ ]:
SESSION_NUM = 0
CALL_KEY    = '6'  # fixed: call_6_qa

# ── resolve call_6 prompt log path ───────────────────────────────────────────
session_dir = GRAPH_PATH.parents[3]
prompt_log  = session_dir / 'prompt_log' / f'session_{SESSION_NUM}'
matches     = [d for d in prompt_log.iterdir() if d.name.startswith(f'call_{CALL_KEY}_')]

if not matches:
    raise FileNotFoundError(f'call_{CALL_KEY}_* 폴더를 찾을 수 없습니다: {prompt_log}')
call_dir = sorted(matches)[0]
calls_path = call_dir / 'calls.jsonl'

# ── load calls + QA results + opposed_implicit_reasoning ─────────────────────
RESULTS_PATH = Path('config_0_outputs_Qwen3-1.7B_opposed/session_0_49/'
                    'results_Qwen3-1.7B_opposed_session_0_49.json')
QA_DATASET_PATH = Path('../dataset/implexconv/ImplexConv_opposed_qa.json')

with open(calls_path) as f:
    records = [json.loads(line) for line in f if line.strip()]
with open(RESULTS_PATH) as f:
    results = json.load(f)
with open(QA_DATASET_PATH) as f:
    qa_dataset = json.load(f)

session_result = next((r for r in results if r['session_id'] == SESSION_NUM), None)
session_qa     = next((d for d in qa_dataset if d['session_id'] == SESSION_NUM), None)
qa_results = session_result.get('qa_results', []) if session_result else []
qa_list    = session_qa['qa'] if session_qa else []

print(f'파일: {calls_path}')
print(f'records: {len(records)}, qa_results: {len(qa_results)}\n')
print(f'\n■ system_prompt\n{rec["system_prompt"]}')

for i, rec in enumerate(records):
    print(f'{"="*80}')
    print(f'[ QA #{i} ]')

    # 매칭되는 qa_result (인덱스 기준)
    if i < len(qa_results):
        qr = qa_results[i]
        q  = qr['question']
        reasoning = next(
            (qa.get('opposed_implicit_reasoning') for qa in qa_list if qa['question'] == q),
            None,
        )
        retrieved_conv_ids = next(
            (qa.get('retrieved_conv_ids') for qa in qa_list if qa['question'] == q),
            None,
        )
        print(f'\n■ question\n{q}')
        print(f'\n■ generated_answer\n{qr["generated_answer"]}')
        print(f'\n■ ground_truth_answer\n{qr["ground_truth_answer"]}')
        print(f'\n■ opposed_implicit_reasoning\n{reasoning if reasoning is not None else "(매칭되는 question 없음)"}')
        print(f'\n■ retrieved_conv_ids: {retrieved_conv_ids if retrieved_conv_ids is not None else "(매칭되는 question 없음)"}')
    else:
        print(f'\n(qa_results[{i}] 없음)')
    
    print(f'\n{"-"*80}')

    # print(f'\n■ system_prompt\n{rec["system_prompt"]}')
    print(f'\n■ user_prompt\n{rec["user_prompt"]}')


파일: config_0_outputs_Qwen3-1.7B_opposed/session_0_49/prompt_log/session_0/call_6_qa/calls.jsonl
records: 5, qa_results: 5


■ system_prompt
You are an assistant who has talked with this user across multiple sessions.
Retrieved memory node types:
- State: a time-bounded user condition (situation, goal, constraint).
- Trait: a stable user characteristic across situations.
- Memory: an episodic summary of a past conversation.
Sections: Current Constraints (HIGH recall-priority states to honor unless overridden), Traits, Challenged Traits (possibly outdated; prefer "shifted to" if listed), Relevant States, Relevant Memories, Recent Conversation.

Use the retrieved memory to tailor your answer. If any item — even one off the question's topic — describes a constraint, situation, or trait that would change a standard answer, incorporate it. Do not give a generic answer when a relevant user circumstance is available.

Answer naturally; do not say "based on what I remember" or similar. Answer i

In [ ]:
SESSION_NUM = 0
CONV_ID     = 67

PROCESSED_PATH = Path('../dataset/implexconv/ImplexConv_opposed_processed.json')

with open(PROCESSED_PATH) as f:
    processed = json.load(f)

session_data = next((d for d in processed if d['metadata']['session_id'] == SESSION_NUM), None)
if session_data is None:
    print(f'session_id={SESSION_NUM} 를 찾을 수 없습니다.')
else:
    turns = [t for t in session_data['conversations'] if t['conv_id'] == CONV_ID]
    if not turns:
        print(f'session_id={SESSION_NUM}, conv_id={CONV_ID} 를 찾을 수 없습니다.')
    else:
        print(f'conv_id = {CONV_ID}\n')

        for t in sorted(turns, key=lambda x: x['turn_id']):
            print(f'{t["speaker"]}: {t["utterance"]}')

conv_id = 67

user: Hi, I'm feeling a bit down today. How can I talk to you about it?
assistant: I'm here to listen and help in any way I can. Please feel free to share whatever is on your mind, and we'll go from there.
user: I don't know... I just feel really overwhelmed, I guess.
assistant: It can be really tough to deal with those feelings. Can you tell me a bit more about what's been going on that's making you feel overwhelmed?
user: It's just been a long week, I think. I'm having trouble sleeping and I just feel really anxious all the time.
assistant: I'm so sorry to hear that. It sounds like you're really struggling right now. Have you noticed any changes at work or in your personal life that might be contributing to these feelings?
user: Actually, yeah. Work has been pretty stressful lately. We're in a busy period and everyone's under a lot of pressure.
assistant: That can be really tough. It sounds like the stress at work might be carrying over into your personal life. Can you 

In [ ]:
GRAPH_PATH_C1 = Path(
    f'config_0_outputs_Qwen3-1.7B_opposed/session_0_49/'
    f'memory_snapshots/session_{SESSION_NUM}/graph/graph.json'
)

with open(GRAPH_PATH_C1) as f:
    G1 = json.load(f)

TYPE_LABEL = {'s': 'state', 'e': 'episode', 't': 'trait'}

print(f'conv_id: {CONV_ID}\n')
for node_type, label in TYPE_LABEL.items():
    matches = [
        n for n in G1['nodes'].values()
        if n['node_type'] == node_type and n.get('conv_id') == CONV_ID
    ]
    print(f'[{label}]')
    if matches:
        for n in matches:
            print("→ "+ n['content'])
    else:
        print('(없음)')
    print()

conv_id: 67

[state]
→ The user is feeling overwhelmed and down today.
→ The user is feeling overwhelmed and anxious due to a long week and difficulty sleeping.
→ The user is experiencing stress and anxiety due to a busy work schedule and feelings of overwhelm.
→ The user feels like they're under a microscope all the time, with every little mistake being scrutinized and causing anxiety.
→ The user is feeling watched and scrutinized, which is making it hard for them to focus and concentrate.
→ The user is feeling stressed and anxious at work due to constant scrutiny and pressure.
→ The user is feeling under a microscope at work, with every little mistake being scrutinized and causing significant anxiety.

[memory]
→ The user shared they are feeling overwhelmed and anxious due to a stressful work environment and lack of sleep, and the assistant offered support by suggesting talking to supervisors or HR and relaxation techniques.

[trait]
→ The user is interested in outdoor activities and